# QA Tasks Aggregated Analysis

## Executive Summary

**Objective:** Aggregate and compare QA performance across all datasets to answer 3 key questions:

### 🎯 Key Questions:
1. **Which is the best phase for each dataset?**
2. **Which is the best model per phase for each dataset?**
3. **What's the best combination (phase + model) for each dataset?**

**Datasets Included:**
- ✅ **DocVQA_mini** (500 samples, 8 phases) - Document Visual QA
- ✅ **InfographicVQA_mini** (500 samples, 11 phases) - Infographic QA
- ⚠️ **dude_mini** (experimental) - Document Understanding
- ⚠️ **chartqapro_mini** (experimental) - Chart QA

**Evaluation Metrics:**
- 🎯 **PRIMARY: GT in Pred** (Ground Truth in Prediction, higher is better)
- **SECONDARY:** ANLS, Exact Match, Substring Match, Cosine Similarity

**QA Strategies:**
- **QA1 (OCR+VLM):** Two-step pipeline with OCR → LLM
- **QA2 (VLM Parse+QA):** Single VLM does both parsing and QA
- **QA3 (Direct VQA):** VLM sees image directly
- **QA4 (Special):** Dataset-specific approaches


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm

# Setup paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import QA metrics
from ocr_vs_vlm.metrics.evaluation_metrics import (
    compute_anls,
    compute_exact_match,
    compute_substring_match,
    compute_ground_truth_in_prediction,
    compute_prediction_in_ground_truth
)
from ocr_vs_vlm.metrics.embedding_cache import EmbeddingCacheManager

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

# Paths
RESULTS_BASE = REPO_ROOT / 'ocr_vs_vlm' / 'results'
EMBEDDINGS_DIR = RESULTS_BASE / '3_embeddings'

print("✅ Libraries loaded successfully!")
print(f"📂 Results base: {RESULTS_BASE}")


✅ Libraries loaded successfully!
📂 Results base: /Users/kenzabenkirane/Documents/GitHub/research-playground/ocr_vs_vlm/results


In [2]:
# QA Dataset configuration
QA_DATASETS = {
    'DocVQA_mini': {
        'path': RESULTS_BASE / '2_clean' / 'DocVQA_mini',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b'],
        'status': 'production'
    },
    'InfographicVQA_mini': {
        'path': RESULTS_BASE / '2_clean' / 'InfographicVQA_mini',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b', 'QA4a', 'QA4b', 'QA4c'],
        'status': 'production'
    },
    'dude_mini': {
        'path': RESULTS_BASE / '2_clean' / 'dude_mini',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b'],
        'status': 'experimental'
    },
    'chartqapro_mini': {
        'path': RESULTS_BASE / '2_clean' / 'chartqapro_mini',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b'],
        'status': 'experimental'
    }
}

def get_phase_strategy(phase: str) -> str:
    """Get strategy group for phase"""
    if phase.startswith('QA1'): return 'QA1 (OCR+VLM)'
    elif phase.startswith('QA2'): return 'QA2 (VLM Parse+QA)'
    elif phase.startswith('QA3'): return 'QA3 (Direct VQA)'
    elif phase.startswith('QA4'): return 'QA4 (Special)'
    return 'Unknown'

print("📁 QA Dataset Configuration:")
for name, config in QA_DATASETS.items():
    status_icon = '✅' if config['status'] == 'production' else '⚠️'
    print(f"  {status_icon} {name}: {len(config['phases'])} phases")


📁 QA Dataset Configuration:
  ✅ DocVQA_mini: 8 phases
  ✅ InfographicVQA_mini: 11 phases
  ⚠️ dude_mini: 8 phases
  ⚠️ chartqapro_mini: 8 phases


In [3]:
def is_valid_row(row, pred_col: str, err_col: Optional[str] = None) -> bool:
    """Check if prediction is valid"""
    pred_value = row[pred_col]
    if pd.isna(pred_value) or str(pred_value).strip() == "":
        return False
    if err_col and err_col in row.index:
        if pd.notna(row[err_col]) and str(row[err_col]).strip() != "":
            return False
    return True

def parse_ground_truths(gt_string) -> List[str]:
    """Parse ground_truths from JSON string"""
    if pd.isna(gt_string):
        return []
    if isinstance(gt_string, list):
        return gt_string
    try:
        return json.loads(gt_string)
    except:
        return [str(gt_string)]

def calculate_qa_metrics(pred: str, ground_truths: List[str], 
                         phase: str, sample_id: str, model: str,
                         emb_manager: EmbeddingCacheManager) -> Dict:
    """Calculate all QA metrics"""
    if pd.isna(pred) or pred == "" or not ground_truths:
        return {
            'gt_in_pred': 0.0,
            'anls': 0.0,
            'exact_match': 0.0,
            'substring_match': 0.0,
            'cosine_similarity': 0.0
        }
    
    pred_str = str(pred)
    cosine_sim = emb_manager.compute_cosine_similarity(
        phase=phase, ground_truth=ground_truths[0],
        prediction=pred_str, sample_id=sample_id, model=model
    )
    
    return {
        'gt_in_pred': compute_ground_truth_in_prediction(pred_str, ground_truths),
        'anls': compute_anls(pred_str, ground_truths, threshold=0.5),
        'exact_match': compute_exact_match(pred_str, ground_truths),
        'substring_match': compute_substring_match(pred_str, ground_truths),
        'cosine_similarity': float(cosine_sim)
    }

print("✅ Helper functions defined")


✅ Helper functions defined


In [ ]:
# Load all QA datasets
all_data = {}
embedding_managers = {}

for dataset_name, config in QA_DATASETS.items():
    print(f"\nLoading {dataset_name}...")
    
    # Initialize embedding manager
    emb_manager = EmbeddingCacheManager(dataset_name, EMBEDDINGS_DIR)
    embedding_managers[dataset_name] = emb_manager
    
    dataset_dfs = {}
    for phase in config['phases']:
        file_path = config['path'] / f"{phase}.csv"
        if file_path.exists():
            df = pd.read_csv(file_path)
            dataset_dfs[phase] = df
            print(f"  ✅ {phase}: {len(df)} samples")
        else:
            print(f"  ⚠️  {phase}: File not found")
    
    all_data[dataset_name] = {
        'config': config,
        'phase_dfs': dataset_dfs
    }

print(f"\n✅ Loaded {len(all_data)} datasets")


In [ ]:
# Calculate QA metrics for all datasets
all_metrics = []

for dataset_name, data in all_data.items():
    if not data['phase_dfs']:
        print(f"⚠️  Skipping {dataset_name} - no data loaded")
        continue
    
    print(f"\nCalculating metrics for {dataset_name}...")
    emb_manager = embedding_managers[dataset_name]
    
    for phase, df in data['phase_dfs'].items():
        pred_cols = [col for col in df.columns if col.startswith('prediction_')]
        
        for pred_col in pred_cols:
            model = pred_col.replace('prediction_', '')
            err_col = f'error_{model}'
            
            # Get valid rows
            valid_rows = [r for _, r in df.iterrows() if is_valid_row(r, pred_col, err_col)]
            
            if not valid_rows:
                continue
            
            # Calculate metrics
            metrics_list = []
            for row in tqdm(valid_rows, desc=f"  {phase}/{model}", leave=False):
                gts = parse_ground_truths(row['ground_truths'])
                m = calculate_qa_metrics(
                    row[pred_col], gts, phase,
                    row['sample_id'], model, emb_manager
                )
                metrics_list.append(m)
            
            # Aggregate
            all_metrics.append({
                'dataset': dataset_name,
                'phase': phase,
                'strategy': get_phase_strategy(phase),
                'model': model,
                'gt_in_pred': np.mean([m['gt_in_pred'] for m in metrics_list]),
                'anls': np.mean([m['anls'] for m in metrics_list]),
                'exact_match': np.mean([m['exact_match'] for m in metrics_list]),
                'substring_match': np.mean([m['substring_match'] for m in metrics_list]),
                'cosine_similarity': np.mean([m['cosine_similarity'] for m in metrics_list]),
                'valid_samples': len(valid_rows)
            })

metrics_df = pd.DataFrame(all_metrics)
print(f"\n✅ Calculated metrics for {len(metrics_df)} combinations")


## 🎯 Section 3: Best Phase per Dataset

**KEY QUESTION #1: Which is the best phase for each dataset?**

This section identifies the best-performing phase for each dataset based on GT in Pred (PRIMARY metric).


In [ ]:
# Identify best phase per dataset
print("="*80)
print("BEST PHASE PER DATASET (by GT in Pred - PRIMARY METRIC)")
print("="*80)

best_phases = []

for dataset in metrics_df['dataset'].unique():
    dataset_data = metrics_df[metrics_df['dataset'] == dataset]
    
    # Average across models per phase
    phase_avg = dataset_data.groupby('phase').agg({
        'gt_in_pred': 'mean',
        'anls': 'mean',
        'exact_match': 'mean'
    }).round(4)
    
    best_phase = phase_avg['gt_in_pred'].idxmax()
    best_metrics = phase_avg.loc[best_phase]
    
    best_phases.append({
        'Dataset': dataset,
        'Best Phase': best_phase,
        'Strategy': get_phase_strategy(best_phase),
        'GT in Pred': best_metrics['gt_in_pred'],
        'ANLS': best_metrics['anls'],
        'Exact Match': best_metrics['exact_match']
    })
    
    print(f"\n{dataset}:")
    print(f"  🏆 Best Phase: {best_phase} ({get_phase_strategy(best_phase)})")
    print(f"  🎯 GT in Pred: {best_metrics['gt_in_pred']:.4f}")
    print(f"     ANLS: {best_metrics['anls']:.4f}, EM: {best_metrics['exact_match']:.4f}")

best_phases_df = pd.DataFrame(best_phases)
print("\n" + "="*80)
print("SUMMARY TABLE:")
display(best_phases_df)


In [ ]:
# Visualize best phase per dataset
fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(best_phases_df))
bars = ax.bar(x, best_phases_df['GT in Pred'], color='steelblue', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f"{row['Dataset']}\n({row['Best Phase']})" 
                     for _, row in best_phases_df.iterrows()], 
                    rotation=45, ha='right')
ax.set_ylabel('GT in Pred (PRIMARY METRIC)', fontweight='bold')
ax.set_title('🏆 Best Phase per Dataset', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 🎯 Section 4: Best Model per Phase per Dataset

**KEY QUESTION #2: Which is the best model per phase for each dataset?**

This section identifies the best-performing model for each dataset-phase combination.


In [ ]:
# Identify best model per phase per dataset
print("="*80)
print("BEST MODEL PER PHASE PER DATASET")
print("="*80)

best_models_per_phase = []

for dataset in metrics_df['dataset'].unique():
    print(f"\n{dataset}:")
    dataset_data = metrics_df[metrics_df['dataset'] == dataset]
    
    for phase in sorted(dataset_data['phase'].unique()):
        phase_data = dataset_data[dataset_data['phase'] == phase]
        best_idx = phase_data['gt_in_pred'].idxmax()
        best = phase_data.loc[best_idx]
        
        best_models_per_phase.append({
            'Dataset': dataset,
            'Phase': phase,
            'Strategy': best['strategy'],
            'Best Model': best['model'],
            'GT in Pred': best['gt_in_pred'],
            'ANLS': best['anls'],
            'Exact Match': best['exact_match']
        })
        
        print(f"  {phase}: {best['model'][:30]}... (GT in Pred={best['gt_in_pred']:.4f})")

best_models_df = pd.DataFrame(best_models_per_phase)
print("\n" + "="*80)
print("SUMMARY TABLE:")
display(best_models_df)


## 🎯 Section 5: Best Combination per Dataset

**KEY QUESTION #3: What's the best combination (phase + model) for each dataset?**

This section identifies the single best phase-model combination for each dataset.


In [ ]:
# Find best combination per dataset
print("="*80)
print("BEST COMBINATION (Phase + Model) PER DATASET")
print("="*80)

best_combinations = []

for dataset in metrics_df['dataset'].unique():
    dataset_data = metrics_df[metrics_df['dataset'] == dataset]
    best_idx = dataset_data['gt_in_pred'].idxmax()
    best = dataset_data.loc[best_idx]
    
    # Get baseline (QA1a) for comparison
    baseline_data = dataset_data[dataset_data['phase'] == 'QA1a']
    if len(baseline_data) > 0:
        baseline_gt = baseline_data['gt_in_pred'].mean()
        improvement = ((best['gt_in_pred'] - baseline_gt) / baseline_gt * 100) if baseline_gt > 0 else 0
    else:
        improvement = 0
    
    best_combinations.append({
        'Dataset': dataset,
        'Best Phase': best['phase'],
        'Strategy': best['strategy'],
        'Best Model': best['model'],
        'GT in Pred': best['gt_in_pred'],
        'ANLS': best['anls'],
        'Exact Match': best['exact_match'],
        'Improvement vs Baseline (%)': round(improvement, 2)
    })
    
    print(f"\n{dataset}:")
    print(f"  🏆 Best Combination: {best['phase']} + {best['model'][:40]}")
    print(f"  🎯 GT in Pred: {best['gt_in_pred']:.4f}")
    print(f"     ANLS: {best['anls']:.4f}, EM: {best['exact_match']:.4f}")
    if improvement != 0:
        print(f"     📈 Improvement vs baseline: {improvement:.2f}%")

best_combinations_df = pd.DataFrame(best_combinations)
print("\n" + "="*80)
print("SUMMARY TABLE:")
display(best_combinations_df)


In [ ]:
# Visualize best combinations
fig, ax = plt.subplots(figsize=(14, 7))

x = range(len(best_combinations_df))
bars = ax.bar(x, best_combinations_df['GT in Pred'], 
              color='forestgreen', alpha=0.8)

ax.set_xticks(x)
labels = []
for _, row in best_combinations_df.iterrows():
    model_short = row['Best Model'][:20] + '...' if len(row['Best Model']) > 20 else row['Best Model']
    labels.append(f"{row['Dataset']}\n{row['Best Phase']} + {model_short}")
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('GT in Pred (PRIMARY METRIC)', fontweight='bold')
ax.set_title('🏆 Best Combination (Phase + Model) per Dataset', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    height = bar.get_height()
    improvement = best_combinations_df.iloc[i]['Improvement vs Baseline (%)']
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}\n({improvement}%)',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Export summary tables
output_dir = RESULTS_BASE / '4_notebooks' / 'output'
output_dir.mkdir(exist_ok=True)

best_phases_df.to_csv(output_dir / 'best_phases_qa.csv', index=False)
best_models_df.to_csv(output_dir / 'best_models_per_phase_qa.csv', index=False)
best_combinations_df.to_csv(output_dir / 'best_combinations_qa.csv', index=False)

print("✅ Summary tables exported:")
print(f"   - {output_dir / 'best_phases_qa.csv'}")
print(f"   - {output_dir / 'best_models_per_phase_qa.csv'}")
print(f"   - {output_dir / 'best_combinations_qa.csv'}")


## 📊 Strategy Analysis

Compare QA strategies (QA1, QA2, QA3, QA4) across datasets.


In [ ]:
# Strategy comparison
print("Strategy Performance Across Datasets:")
print("="*80)

strategy_summary = metrics_df.groupby(['dataset', 'strategy']).agg({
    'gt_in_pred': 'mean',
    'anls': 'mean',
    'exact_match': 'mean'
}).round(4)

display(strategy_summary)

# Overall strategy ranking
overall_strategy = metrics_df.groupby('strategy').agg({
    'gt_in_pred': 'mean',
    'anls': 'mean',
    'exact_match': 'mean'
}).round(4).sort_values('gt_in_pred', ascending=False)

print("\nOverall Strategy Ranking:")
display(overall_strategy)


## 📊 Final Summary

This notebook has answered the 3 key questions:

### ✅ Question 1: Best Phase per Dataset
Identified the optimal phase for each dataset based on GT in Pred (PRIMARY metric).

### ✅ Question 2: Best Model per Phase per Dataset
Identified the best-performing model for each dataset-phase combination.

### ✅ Question 3: Best Combination per Dataset
Identified the single best phase-model combination per dataset with improvement percentages.

**Exported Files:**
- `best_phases_qa.csv` - Best phase per dataset
- `best_models_per_phase_qa.csv` - Best model for each dataset-phase
- `best_combinations_qa.csv` - Best phase-model per dataset

**Key Findings:**
- QA strategies show varying performance across datasets
- Direct VQA (QA3) often performs well for visual reasoning
- OCR+VLM pipeline (QA1) excels for text-heavy documents

**Next Steps:**
- Review [parsing_analysis.ipynb](parsing_analysis.ipynb) for parsing task analysis
- Use results for production model selection
- Refer to individual dataset notebooks in `by_dataset/` for detailed analysis
